<a href="https://colab.research.google.com/github/arghavanedalat/arghavanedalat.github.io/blob/master/Another_copy_of_monotonicity_on_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from itertools import product
from scipy.optimize import minimize_scalar, root_scalar , root

In [2]:
#extracting relevant columns from each dataset
person = pd.read_csv("person.csv", sep=';',low_memory=False)[['PECH','ECH', 'P4']]
menage = pd.read_csv("menage.csv", sep=';',low_memory=False)[['ECH', 'NPC']]
deplacement = pd.read_csv("deplacement.csv", sep=';',low_memory=False)[['PECH', 'D9', 'D8C', 'DIST', 'MODP']]

new = pd.merge(person,menage,how = 'inner',on = 'ECH')
merged_data = pd.merge(new,deplacement,how = 'inner',on = 'PECH')
#creating the new column based on MODP values
merged_data['mode_type'] = np.where(merged_data['MODP'].isin([21, 22, 61, 81]), 0, 1)

#load dataset
X = merged_data.drop(columns=['MODP','PECH','ECH', 'mode_type'])
y = merged_data['mode_type']

#remove missing values
X = X.apply(lambda col: col.fillna(col.median()) if col.dtype != 'O' else col)
y = y.dropna()
X = X.loc[y.index]

#split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [3]:
#apply logit model
logistic_reg = LogisticRegression(max_iter=1000)
logistic_reg.fit(X_train, y_train)

#extract coefficients
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': logistic_reg.coef_[0]})
intercept = logistic_reg.intercept_[0]

#extract utility function
def compute_utility(X, model):
    coefficients = model.coef_[0]
    intercept = model.intercept_[0]
    return intercept + np.dot(X, coefficients)

logit_utility_train = compute_utility(X_train, logistic_reg)
logit_utility_test = compute_utility(X_test, logistic_reg)

#computing the probabilit sigmoid
logit_prob_train = 1/ (1 + (np.exp(-logit_utility_train)))
logit_prob_test = 1 / (1 + np.exp(-logit_utility_test))

print(logistic_reg.coef_[0])
print(np.round(logistic_reg.coef_[0]).astype(int))

[-2.53805902e-02 -6.84257698e-02 -3.83851208e+00  2.29703767e-02
 -8.17922711e-06]
[ 0  0 -4  0  0]


In [4]:
#SLSQP min for finding ext values

x_min = X_train.min()
x_max = X_train.max()

#extending range
x_min_extend = x_min - 0.1 * (x_max - x_min)
x_max_extend = x_max + 0.1 * (x_max - x_min)

#bound of the optimization
bounds = list(zip(x_min_extend, x_max_extend))

#initial guess
x0_new = np.array([np.random.uniform(low, high) for low, high in zip(x_min_extend, x_max_extend)])

def objective_min(x):
    u = intercept + np.dot(x, logistic_reg.coef_[0])
    return 1 / (1 + np.exp(-u))

#applying the constraint
def monotonic_constraints_min(coefficients, x_min, x_max):
    constraints = []
    for i, coeff in enumerate(coefficients):
        if coeff > 0:
            constraints.append({'type': 'ineq', 'fun': lambda x, i=i:  x_min[i]-x[i]})
        elif coeff < 0:
            constraints.append({'type': 'ineq', 'fun': lambda x, i=i:  x[i] - x_max[i]})
    return constraints


coeffs = logistic_reg.coef_[0]
custom_constraints_min = monotonic_constraints_min(coeffs, x_min_extend.values, x_max_extend.values)

result_min = minimize(objective_min, x0_new, constraints=custom_constraints_min, method='SLSQP')
result_min.x[result_min.x < 0] = 0

print(result_min.x)
print(result_min.fun)

x_min_rounded = np.round(result_min.x).astype(int)
print(x_min_rounded)

#SLSQP max for finding ext values

def objective_max(x):
    u = intercept + np.dot(x, logistic_reg.coef_[0])
    return -1 / (1 + np.exp(-u))

#applying the constraint
def monotonic_constraints_max(coefficients, x_min, x_max):
    constraints = []
    for i, coeff in enumerate(coefficients):
        if coeff > 0:
            constraints.append({'type': 'ineq', 'fun': lambda x, i=i:  x[i] - x_max[i]})
        elif coeff < 0:
            constraints.append({'type': 'ineq', 'fun': lambda x, i=i:  x_min[i]-x[i]})
    return constraints


coeffs = logistic_reg.coef_[0]
custom_constraints_max = monotonic_constraints_max(coeffs, x_min_extend.values, x_max_extend.values)

result_max = minimize(objective_max, x0_new, constraints=custom_constraints_max, method='SLSQP')
result_max.x[result_max.x < 0] = 0

print(result_max.x)
print(result_max.fun)

x_max_rounded = np.round(result_max.x).astype(int)
print(x_max_rounded)

[1.04500101e+02 6.50000918e+00 5.50000914e+00 0.00000000e+00
 2.07003532e+06]
1.6267571570801766e-18
[    105       7       6       0 2070035]
[2.50e+00 5.00e-01 0.00e+00 1.32e+03 0.00e+00]
-0.9999999999999998
[   2    1    0 1320    0]


In [ ]:
from scipy import optimize
obj = logistic_reg.coef_[0]
intercept = logistic_reg.intercept_[0]

bounds = []
for i in range(len(obj)):
    lower_bound = X_train.iloc[:, i].min()*0.75
    upper_bound = X_train.iloc[:, i].max() *1.25
    bounds.append((lower_bound, upper_bound))

result_max = optimize.linprog(c=-obj, bounds=bounds, method='highs')
result_min = optimize.linprog(c=obj, bounds=bounds, method='highs')

print(result_max.x)
print(result_min.x)


[8.25e+00 7.50e-01 0.00e+00 1.50e+03 0.00e+00]
[1.2000000e+02 7.5000000e+00 6.2500000e+00 0.0000000e+00 2.3523125e+06]


In [ ]:
#quadratic programming


In [44]:
monotonicity = [False, False, False, True, False]
#nonlinear utility and optimal parameters
def a_i(c, xumin, xumax):
  return -(np.exp(c*xumin))/(np.exp(c*xumax)-np.exp(c*xumin))

def b_i(c, xumin, xumax):

  return 1 / (np.exp(c * xumax) - np.exp(c * xumin))
#k_i
def logit_prob(x, coef, intercept):
    u =intercept+np.dot(x, coef)
    return 1/(1+np.exp(-u))

def k_i(coef,intercept, x_min,x_max, monotonicity):
    k = []
    for i in range(len(coef)):
        x = np.zeros(len(coef))
        for j in range(len(coef)):
            if i==j:
              if monotonicity[j]== True:
                x[j] =x_max[j]
              else:
                 x[j] =x_min[j]
            else:
              if monotonicity[j]== True:
                x[j]=x_min[j]
              else:
                x[j]=x_max[j]
        k_i = logit_prob(x, coef, intercept)
        k.append(k_i)
    return np.array(k)

#K
def g(K,ki):
    prod = 1
    for i in ki:
        t = (1+K * i)
        prod *= t
    result = prod-(1+K)
    return result

def solve_K(ki, bracket=(-10,10)):
    def g(K):
        prod = 1.0
        for k in ki:
            prod *= (1 + K * k)
        return prod - (1 + K)
    result = root_scalar(g, method='brentq', bracket=bracket)
    return result.root

#y
#for y compute the logit of the dataset for when for each variable we have all
#the values and fix the other values as the value that made the global U to 1 which is
#equal to result_max
#find the values of each feature and copy into a new var
#we have to fix others values as their value in xmax and change j value at the same time

def y(xmax, x, coef, intercept):
    new_dataset=[]
    logit_outputs=np.zeros((x.shape[0],x.shape[1]))
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            unique_values= np.unique(x.iloc[:,j])
            for val in unique_values:
                x_new_sample =xmax.copy()
                x_new_sample[j] = val
                logit_outputs[i,j] = logit_prob(x_new_sample, coef, intercept)

    return logit_outputs
#c
#forexample for c1 we try to find it by minimizing the loss function which is the square of difference of y1/k1 and u1
# Compute y_i for each feature, fixing others at result_max


#y
# def y(xmax,x, coef, intercept):
#     logit_outputs = np.zeros((x.shape[0], x.shape[1]))
#     for i in range(x.shape[0]):
#         for j in range(x.shape[1]):
#             x_modified = x.iloc[i].copy()
#             x_modified[j] = xmax[j]
#             logit_outputs[i, j] = logit_prob(x_modified, coef, intercept)
#     return logit_outputs

y2 = y(x_max_rounded,X_train,coeffs,intercept)
print(y2)

#c
def compute_utility(c,x,xumin,xumax):
    a = a_i(c,xumin,xumax)
    b = b_i(c,xumin,xumax)
    u = a + b*np.exp(c*x)
    return u


k_val = k_i(coeffs,intercept,x_min_rounded,x_max_rounded,monotonicity)

K = solve_K(k_val)
y_teacher = y(x_max_rounded,X_train,coeffs,intercept)

c_val = []
a_val = []
b_val = []

#extremum points
xumin = x_min_rounded
xumax = x_max_rounded

for j in range(X_train.shape[1]):
    xj = X_train.iloc[:,j].values
    yj = y_teacher[:,j]/k_val[j]

    def loss_fnc(c):
        bj = b_i(c,xumin[j],xumax[j])
        aj = a_i(c,xumin[j],xumax[j])
        uj = aj+bj*np.exp(c*xj)
        return np.mean((yj-uj)**2)


    res = minimize_scalar(loss_fnc,bounds=(-10,10),method='bounded')
    c_opt = res.x
    c_val.append(c_opt)
    b_opt = b_i(c_opt,xumin[j],xumax[j])
    a_opt = a_i(c_opt,xumin[j],xumax[j])
    a_val.append(a_opt)
    b_val.append(b_opt)
c_val= np.array(c_val)
a_val= np.array(a_val)
b_val= np.array(b_val)



KeyboardInterrupt: 

In [45]:
print(K)
print(k_val)
print(c_val)
print(b_val)
print(a_val)

-2.9060749463492623


NameError: name 'k_val' is not defined

In [25]:
from sklearn.metrics import accuracy_score

def compute_student_utility(X, c_val, a_val, b_val):
    n_samples, n_features = X.shape
    utilities = np.zeros((n_samples, n_features))
    for j in range(n_features):
        cj = c_val[j]
        aj = a_val[j]
        bj = b_val[j]
        xj = X.iloc[:, j].values
        utilities[:, j] = aj + bj * np.exp(cj * xj)
    return utilities

def predict_student(X, c_val, a_val, b_val, K):
    utilities = compute_student_utility(X, c_val, a_val, b_val)
    product = np.prod(1 + K * utilities, axis=1)
    y_pred_prob = (product - 1) / K
    return y_pred_prob

y_pred_prob = predict_student(X_test, c_val, a_val, b_val, K)
y_pred_binary = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred_binary)
print(acc)


0.5320522558104208


In [46]:
#find the values of each feature and copy into a new var
new_feats = []
for i in range(X_train.shape[0]):
    new_feat = X_train.iloc[i]
    new_feats.append(new_feat)
  # for j in range(x.shape[1]):
  #   unique_i = np.unique(x[i])
  #   x_new_sample = x.iloc[i].copy()